# Lightcone Analysis

Reads in lightcone and checks for places where dark photon conversions occur. Then incorporates these effects and produces a power spectrum

In [ ]:
import os
import sys
sys.path.append("../")

sys.path.append("../21cmfast_sim/")
import numpy as np
from astropy.cosmology import Planck18
import py21cmfast as p21c
from scipy.spatial.transform import Rotation
from astropy import units as un

# WDIR = os.environ['DM21CM_DIR']
# sys.path.append(WDIR)
# from dm21cm.evolve import evolve

import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib import colormaps as cms
import matplotlib.pylab as pylab

# mpl.rc_file(f"{WDIR}/matplotlibrc")

# from custom_injection_def import DarkPhotInjection
# import darkhistory.dark_photon as dp
# import darkhistory.physics as phys
from itertools import combinations
from powerbox.tools import get_power
import physics as phys
import pickle
import h5py

from astropy.cosmology import z_at_value
import tqdm

import scipy.interpolate as interpolate
# plt.style.use('science')
import scipy.integrate as integrate
import scipy.stats as stats
import scipy.stats.sampling as sampling
import scipy.fft as fft

from powerbox import PowerBox


%load_ext autoreload
%autoreload 2
%matplotlib inline

from galaxy_survey import *
pylab.rcParams.update(params)
cols_default = plt.rcParams['axes.prop_cycle'].by_key()['color']

In [ ]:
cache_name = f'/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/v4_lightcones/'
# cache_name = '/home/bakerem/dark_photon_21cm_constraints/halo_data/v4_lightcones/'
# lc_name  = 'LightCone_z5.5_HIIDIM=48_BOXLEN=64.0_r12345.h5'
lc_name = "LightCone_z5.5_HIIDIM=200_BOXLEN=300.0_r3843290498042390.h5"
# lightconer_name = "lightconer_seed12345.pkl"
lightconer_name = "lightconer_seed3843290498042390.pkl"
point_source_dir = "/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pointsource_sky_maps/"
roman = Survey("Roman", "Lya", 
        cache=cache_name, 
        lc_name=lc_name, 
        lightconer_name=lightconer_name, 
        nside=2048, 
        # foregrounds_path="/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_no_pt_srcs/all_foregrounds.npy",
        foregrounds_path="/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_foregrounds_no_pt_srcs_low_freq/",
        clean_map_loc="/projectnb/darkcosmo/dark_photon_project/21cmfast_cache/pyilc_foregrounds_no_pt_srcs_low_freq/clean_needlet_ILC_map_subaru_20deg_LOFARthermal_mK.fits",
        use_pixel_ilc=False,
        )

In [ ]:
roman.get_bt_plus_dp(mA=3e-14, epsilon=1e-6);

In [ ]:
def compute_2d_power(
   box,
   length,
   n_psbins,
   log_bins=True,
   ignore_kperp_zero=True,
):
    """
    Convenience function for computing the power spectrum of a 3D box that wraps get_power from powerbox.  This code is borrowed from the example
    notebook in the 21cmFAST documentation. get_power takes a 3D box and Fourier transforms it and then performs a spherical average in $k$-space. 
    """
    # Determine the weighting function required from ignoring k's.
    k_weights = np.ones(box.shape[:2], dtype=int)
    n0 = k_weights.shape[0]
    n1 = k_weights.shape[-1]

    if ignore_kperp_zero:
        k_weights[n0 // 2, n0 // 2] = 0

    res = get_power(
        box,
        boxlength=length,
        # deltax2=box,
        # boxlength2=length
        bins=n_psbins,
        bin_ave=False,
        get_variance=False,
        log_bins=log_bins,
        k_weights=k_weights,
        # bins_upto_boxlen=True,
        res_ndim=2, 
    )

    res = list(res)
    k = res[1]
    if log_bins:
        k = np.exp((np.log(k[1:]) + np.log(k[:-1])) / 2)
    else:
        k = (k[1:] + k[:-1]) / 2

    res[1] = k
    return res

def powerspectra_2d(brightness_temp, n_psbins=25, nchunks=10, min_k=0.1, max_k=1.0, logk=True):
    """
    This function wraps compute_power to compute the power spectrum of many chunks of a single
    lightcone and returns the dimensionless power spectrum. 
    """
    data = []
    n_slices = roman.rs_array.shape[0]
    chunk_indices = list(range(0,n_slices,round(n_slices / nchunks)))

    if len(chunk_indices) > nchunks:
        chunk_indices = chunk_indices[:-1]
    chunk_indices.append(n_slices-1)

    for i in tqdm(range(nchunks)):
        start = chunk_indices[i]
        end = chunk_indices[i + 1]
        cell_size = roman.box_len / roman.hii_dim
        chunklen = (end - start) * cell_size
        comoving_size = np.max(roman.lat) * roman.cosmo.comoving_distance(np.average([roman.rs_array[start], roman.rs_array[end-1]])).value
        power, k_perp, k_parallel = compute_2d_power(
            brightness_temp[:, :, start:end],
            (comoving_size, comoving_size, chunklen),
            n_psbins,
            log_bins=logk,
        )
        # k = np.concatenate([k_perp, k_parallel])
        k_mag = np.sqrt(k_perp[:, None]**2 + k_parallel[0][None, :]**2)
        data.append({"k_perp": k_perp, "k_par":k_parallel[0], "P": power, "delta": k_mag * power/ (2*np.pi**2)})
    return data, chunk_indices

In [ ]:
powerspectra_2d_results, chunk_indices = powerspectra_2d(roman.dp_lightcone)

In [ ]:
import astropy.constants as const
chunk_zs = roman.rs_array[np.array(chunk_indices)]
average_zs = (chunk_zs[:-1] + chunk_zs[1:])/2
slopes = (roman.cosmo.H(average_zs) * roman.cosmo.comoving_distance(average_zs) / (const.c * (1+average_zs))).to(un.dimensionless_unscaled)


In [ ]:
# represents optimistic removal
chunk = 8
print(average_zs[chunk])
k_perp = powerspectra_2d_results[chunk]["k_perp"]
k_par  = powerspectra_2d_results[chunk]["k_par"]
wedge = 0.1 * roman.cosmo.H0.value/100 + slopes[chunk]/2 * k_perp
plt.pcolormesh(k_perp, k_par[k_par.shape[0]//2+1:], np.swapaxes(powerspectra_2d_results[chunk]["delta"][:,k_par.shape[0]//2+1:], 0,1), 
            norm="log",
            cmap="plasma",
            )
plt.fill_between(k_perp, wedge,  color="black", alpha=0.35)

plt.xscale("log")
plt.yscale("log")
cbar = plt.colorbar()
cbar.set_label(r"$T_{\gamma,0}^2 \Delta^2_{\gamma \to A'}$ [mK$^2$]")
plt.xlabel(r"$k_{\perp}$ [Mpc$^{-1}$]")
plt.ylabel(r"$k_{\parallel}$ [Mpc$^{-1}$]")
plt.ylim(0.01, 2)
plt.xlim(0.025, 7.6)
plt.savefig("plots/21cm_ps.pdf", bbox_inches="tight")
plt.savefig("plots/21cm_ps.png", dpi=600, bbox_inches="tight")
